In [ ]:
# IMPLEMENTACIÓN DEL MODELO EN PRODUCCIÓN

print("="*70)
print("🚀 IMPLEMENTACIÓN EN PRODUCCIÓN - SISTEMA DE ALERTA TEMPRANA")
print("="*70)
print("Objetivo: Cargar el modelo RF Mejorado entrenado y usarlo para predecir")
print("          rotación en nuevos datos, aplicando el umbral óptimo (0.35)")
print("="*70)

# 1. IMPORTAR LIBRERÍAS
print("\n📦 IMPORTANDO LIBRERÍAS...")
import pandas as pd
import numpy as np
import joblib
import pickle
import json
import os
from datetime import datetime

print("✅ Librerías importadas")

# 2. CONFIGURACIÓN
print("\n⚙️  CARGANDO CONFIGURACIÓN...")
model_dir = "../models"

# Buscar el archivo de configuración más reciente
archivos_config = [f for f in os.listdir(model_dir) if f.startswith('configuracion_rf_mejorado_') and f.endswith('.pkl')]
if not archivos_config:
    print("❌ No se encontró archivo de configuración")
    # Usar configuración por defecto
    configuracion = {
        'umbral_recomendado': 0.35,
        'interpretacion_rrhh': 'Umbral por defecto - revisar configuración'
    }
else:
    # Tomar el más reciente (por timestamp)
    archivos_config.sort(reverse=True)
    config_path = os.path.join(model_dir, archivos_config[0])
    
    with open(config_path, 'rb') as f:
        configuracion = pickle.load(f)
    
    print(f"✅ Configuración cargada: {archivos_config[0]}")
    print(f"   • Umbral recomendado: {configuracion.get('umbral_recomendado', 0.35)}")
    print(f"   • Recall esperado: {configuracion.get('recall_esperado', 'N/A')}")
    print(f"   • Fecha entrenamiento: {configuracion.get('fecha_entrenamiento', 'N/A')}")

# 3. CARGAR MODELO
print("\n🤖 CARGANDO MODELO RF MEJORADO...")
modelo_path = os.path.join(model_dir, "randomforest_mejorado.pkl")

if not os.path.exists(modelo_path):
    print(f"❌ Modelo no encontrado: {modelo_path}")
    print("   Buscando alternativa...")
    # Buscar cualquier modelo RF mejorado
    archivos_modelo = [f for f in os.listdir(model_dir) if f.startswith('randomforest_mejorado_') and f.endswith('.pkl')]
    
    if archivos_modelo:
        archivos_modelo.sort(reverse=True)
        modelo_path = os.path.join(model_dir, archivos_modelo[0])
        print(f"   ✅ Modelo alternativo encontrado: {archivos_modelo[0]}")
    else:
        print("❌ No se encontró ningún modelo RF Mejorado")
        raise FileNotFoundError("Modelo no encontrado")

# Cargar el modelo
modelo = joblib.load(modelo_path)
print(f"✅ Modelo cargado: {os.path.basename(modelo_path)}")
print(f"   • Características esperadas: {configuracion.get('caracteristicas', 'N/A')}")

# 4. CARGAR NUEVOS DATOS (SIMULADO - CAMBIA ESTO)
print("\n📁 CARGANDO DATOS PARA PREDECIR...")
print("⚠️  NOTA: Este es un ejemplo. Cambia la ruta para tus datos reales")

# Opción A: Datos de ejemplo (simulados)
def cargar_datos_ejemplo():
    """Crea datos de ejemplo para demostración"""
    print("   Generando datos de ejemplo...")
    n_empleados = 50
    n_features = configuracion.get('caracteristicas', 48)
    
    # Datos aleatorios (en producción cargarías datos reales)
    datos_ejemplo = pd.DataFrame(
        np.random.randn(n_empleados, n_features),
        columns=[f'feature_{i}' for i in range(n_features)]
    )
    
    # Añadir ID de empleado
    datos_ejemplo.insert(0, 'employee_id', range(1000, 1000 + n_empleados))
    
    print(f"   ✅ {n_empleados} empleados simulados, {n_features} características")
    return datos_ejemplo

# Opción B: Cargar datos reales (COMENTADO - DESCOMENTA Y AJUSTA)
# datos_path = "../XXXXXXXXXXX.csv"
# if os.path.exists(datos_path):
#     nuevos_datos = pd.read_csv(datos_path)
#     print(f"✅ Datos cargados: {len(nuevos_datos)} empleados")
# else:
#     print(f"⚠️  Archivo no encontrado: {datos_path}")
#     print("   Usando datos de ejemplo...")
#     nuevos_datos = cargar_datos_ejemplo()

# Por ahora, usamos datos de ejemplo
nuevos_datos = cargar_datos_ejemplo()

# 5. PREPARAR DATOS PARA PREDICCIÓN
print("\n🔧 PREPARANDO DATOS...")
# Separar ID si existe
if 'employee_id' in nuevos_datos.columns:
    ids = nuevos_datos['employee_id']
    X_pred = nuevos_datos.drop('employee_id', axis=1)
else:
    ids = pd.Series(range(len(nuevos_datos)), name='employee_id')
    X_pred = nuevos_datos.copy()

print(f"   • Empleados a evaluar: {len(X_pred)}")
print(f"   • Características: {X_pred.shape[1]}")

# 6. HACER PREDICCIONES
print("\n🎯 REALIZANDO PREDICCIONES...")
umbral = configuracion.get('umbral_recomendado', 0.35)

# Obtener probabilidades
probabilidades = modelo.predict_proba(X_pred)[:, 1]

# Aplicar umbral
predicciones = (probabilidades > umbral).astype(int)

print(f"✅ Predicciones completadas con umbral {umbral}")
print(f"   • Alertas generadas: {sum(predicciones)} de {len(predicciones)} ({sum(predicciones)/len(predicciones):.1%})")

# 7. CREAR REPORTE
print("\n📊 GENERANDO REPORTE PARA RRHH...")

# DataFrame con resultados
resultados = pd.DataFrame({
    'employee_id': ids.values if hasattr(ids, 'values') else ids,
    'probabilidad_rotacion': probabilidades,
    'alerta_rotacion': predicciones,
    'nivel_riesgo': pd.cut(probabilidades, 
                          bins=[0, 0.3, 0.5, 0.7, 1.0],
                          labels=['Bajo', 'Moderado', 'Alto', 'Crítico'])
})

# Ordenar por probabilidad descendente
resultados = resultados.sort_values('probabilidad_rotacion', ascending=False)

print("\n" + "="*70)
print("📈 RESUMEN DE ALERTAS")
print("="*70)

print(f"\n🔔 ALERTAS DE ROTACIÓN ({umbral}): {sum(predicciones)} empleados")
print("-" * 50)

# Mostrar empleados con mayor riesgo
n_top = min(10, sum(predicciones))
if n_top > 0:
    print(f"\n🚨 TOP {n_top} EMPLEADOS CON MAYOR RIESGO:")
    top_riesgo = resultados[resultados['alerta_rotacion'] == 1].head(n_top)
    
    for idx, row in top_riesgo.iterrows():
        print(f"   • ID {row['employee_id']}: {row['probabilidad_rotacion']:.1%} riesgo ({row['nivel_riesgo']})")

print(f"\n📋 DISTRIBUCIÓN DE RIESGO:")
distribucion = resultados['nivel_riesgo'].value_counts().sort_index()
for nivel, count in distribucion.items():
    porcentaje = count / len(resultados) * 100
    print(f"   • {nivel}: {count:3d} empleados ({porcentaje:5.1f}%)")

print(f"\n📊 ESTADÍSTICAS:")
print(f"   • Probabilidad promedio: {probabilidades.mean():.1%}")
print(f"   • Probabilidad máxima: {probabilidades.max():.1%}")
print(f"   • Probabilidad mínima: {probabilidades.min():.1%}")

# 8. GUARDAR RESULTADOS
print("\n💾 GUARDANDO RESULTADOS...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

# Guardar resultados completos
output_path = os.path.join(output_dir, f"alertas_rotacion_{timestamp}.csv")
resultados.to_csv(output_path, index=False)
print(f"✅ Resultados guardados: {output_path}")

# Guardar solo alertas (para RRHH)
alertas_path = os.path.join(output_dir, f"alertas_prioritarias_{timestamp}.csv")
alertas = resultados[resultados['alerta_rotacion'] == 1].copy()
alertas.to_csv(alertas_path, index=False)
print(f"✅ Alertas prioritarias guardadas: {alertas_path} ({len(alertas)} empleados)")

# 9. INTERPRETACIÓN PARA RRHH
print("\n" + "="*70)
print("👥 INTERPRETACIÓN PARA EL DEPARTAMENTO DE RRHH")
print("="*70)

print(f"""
ACCIONES RECOMENDADAS:

1. 📅 PRIORIZAR ENTREVISTAS:
   • Programar entrevistas con los {len(alertas)} empleados alertados
   • Comenzar por los {min(5, len(alertas))} con mayor probabilidad

2. ⏰ ESTIMACIÓN DE CARGA DE TRABAJO:
   • Entrevistas necesarias: {len(alertas)} empleados
   • Tiempo estimado: {len(alertas) * 30} minutos (30 min por entrevista)
   • Equivalente a: {len(alertas) * 30 / 60:.1f} horas-hombre

3. 🎯 FOCO DE LAS ENTREVISTAS:
   • Investigar: Sobrecarga laboral, agotamiento (burnout)
   • Evaluar: Satisfacción con cultura organizacional
   • Explorar: Oportunidades de desarrollo interno
   • Verificar: Nivel de compromiso (engagement)

4. 📈 METAS DEL SISTEMA:
   • Detección esperada: {configuracion.get('recall_esperado', 0.787)*100:.1f}% de casos reales
   • Precisión esperada: {configuracion.get('precision_esperada', 0.336)*100:.1f}% (1 de cada 3 alertas será real)
   • Casos que podrían escaparse: {configuracion.get('falsos_negativos_esperados', 10)} empleados

RECUERDA:
• Este es un sistema de ALERTA TEMPRANA, no de diagnóstico definitivo
• Combina estas alertas con tu conocimiento cualitativo del equipo
• La intervención proactiva es más económica que la rotación no detectada
""")

print("\n" + "="*70)
print("✅ IMPLEMENTACIÓN COMPLETADA")
print("="*70)
print(f"📅 Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Total empleados evaluados: {len(resultados)}")
print(f"🔔 Alertas generadas: {len(alertas)}")
print(f"💾 Archivos guardados en: {output_dir}")
print("="*70)